# Running Log in Cambridge
My goal is to create an html visualisation of a map of Cambridge, where I can add the itinerary of every run I do in cambridge in 2026 to see how it populates.

I am using the folium Python module, and taking inspiration from [this Alexandre Donciu-Julin blog](https://dev.to/alexdjulin/my-runing-map-using-python-and-folium-2h2m)

I will extract the GPX files from Strava

In [2]:
#Setup
import folium
import webbrowser
import os
import gpxpy
import pandas as pd
import math
import folium.plugins as plugins


In [4]:
#First, create the map centered in Cambridge
#The latitude of Cambridge, UK is 52.205276, and the longitude is 0.119167
run_map = folium.Map(location=[52.205276, 0.119167], tiles=None, zoom_start=13)

# add Openstreetmap layer
folium.TileLayer("CartoDB Voyager", name='OpenStreet Map').add_to(run_map)

# save and open map
run_map.save('run_map.html')
#webbrowser.open('file://' + os.path.realpath('run_map.html'))

Folium 'Tile Layer types', i.e. what the map looks like
- "openstreetmap" Standard, with path types
- "CartoDB Positron" Is very minimalist, almost grayscale
- "CartoDB Voyager" More colourull but still very mininmalist (my favorite)
- More: https://leaflet-extras.github.io/leaflet-providers/preview/

In [ ]:
#Do not put the GPS coordinates to the general area of start of most of these runs for privacy

#Hide Latitude > 52.191147 & < 52.192247
#.    & Longitute > 0.124027 & < 0.134040

# forbidden_rect = folium.PolyLine([([52.192247, 0.124027]),
#                               ([52.191147, 0.124027]),
#                               ([52.191147, 0.134040]),
#                               ([52.192247, 0.134040]),
#                               ([52.192247, 0.124027])],
#                             color='gray', weight=5, opacity=0.85)
# forbidden_rect.add_to(run_map)

# #Test rectangle border
# latitude_list = [52.18,52.19,52.195,52.20,52.18,52.1912]
# longitude_list = [0.122,0.123,0.124,0.125,0.1245,0.127]

# for i in range(0,len(latitude_list)):
#   latitude = latitude_list[i]
#   longitude = longitude_list[i]

#   long_lat = str(latitude) + ',' + str(longitude)


#   iframe = folium.IFrame(long_lat)
#   popup = folium.Popup(iframe, min_width="100", max_width="100")
  
#   if not ((52.191147 < latitude < 52.192247) and (0.124027 < longitude < 0.134040)):
#     folium.Marker(location=[latitude, longitude],popup = popup, icon = folium.Icon(color='blue')).add_to(run_map)
#   else:
#     print(i)
#     folium.Marker(location=[latitude, longitude],popup = popup, icon = folium.Icon(color='red')).add_to(run_map)

# run_map.save('run_map.html')

#webbrowser.open('file://' + os.path.realpath('run_map.html'))


5


In [5]:
#Loop through each row in the dataframe
df = pd.read_csv('cambridge_colleges.csv')

for i,row in df.iterrows():
    #Setup the content of the popup
    iframe = folium.IFrame(str(row["College"]))
    
    #Initialise the popup using the iframe
    popup = folium.Popup(iframe, min_width="100", max_width="100")
    
    #Add each row to the map
    #Red teardrops markers
    #folium.Marker(location=[row['Latitude'],row['Longitude']],popup = popup, icon = folium.Icon(color='red', icon='')).add_to(run_map)
    #Red circles as markers    
    folium.Marker(location=[row['Latitude'],row['Longitude']],popup = popup, 
                      icon = plugins.BeautifyIcon(icon="",icon_shape="circle", border_color='red',background_color='red',number=i+1,text_color='white')).add_to(run_map)
run_map.save('run_map.html')

In [6]:

# parse gpx file
gpx_file = 'run_gpx_files/Afternoon_Run.gpx'
gpx = gpxpy.parse(open(gpx_file))
track = gpx.tracks[0]
segment = track.segments[0]

# load coordinate points
points = []
for track in gpx.tracks:
    for segment in track.segments:
        step = 10
        for point in segment.points[::step]:
            ##Add an if statement to not add any points in the forbidden rectangle
            if not ((52.191147 < point.latitude < 52.192247) and (0.124027 < point.longitude < 0.134040)):
                points.append(tuple([point.latitude, point.longitude]))

# add segments to the map
folium_gpx = folium.PolyLine(points, color='red', weight=5, opacity=0.85).add_to(run_map)

# add the gpx trace to our marathon group
folium_gpx.add_to(run_map)
run_map.save('run_map.html')


In [7]:
webbrowser.open('file://' + os.path.realpath('run_map.html'))

True